In [1]:
import pickle
with open("4_with_nc.pkl", "rb") as f:
    df = pickle.load(f)

Input via MODIS NPP + LC info

In [2]:
RS_ef_HI = {
    'A': (      0.00,                    # artificial land: no biomass
            0.00,                        # no roots
            0.00 ),                      # no harvest
    'B': (      0.125,               # cropland: 0.1 to 0.5, cereals ~0.15
            0.965,                  # typical for crops (shallow roots)
            0.5 ),                # grain crops
    'C': (      0.25,             # global forest average
            0.92,                  # deeper rooted than crops
            0.50 ),                     # timber fraction of aboveground biomass
    'D': (      1.68,             # shrubland global average
            0.97,                  # intermediate (deeper than grass)
            0.33 ),                     # non‑crop removal fraction
    'E': (      5.5,                 # highest R:S of any biome
            0.98,                  # very shallow roots (β near 1)
            0.33 ),                     # grazing / biomass removal
    'H': (      0.00,                    # bare land / sparse vegetation
            0.00,                       # no root system
            0.00 ),                     # no harvest
    'F': (      1.0,                 # wetland plants (variable)
            0.965,                  # often shallow‑rooted
            0.33 )                      # harvest similar to grassland
}

# check data:
# Mokany, K., Raison, R.J. and Prokushkin, A.S. (2006). Critical analysis of root : shoot ratios in terrestrial biomes. Global Change Biology, 12(1), pp.84–96.
# Schenk, H.J. and Jackson, R.B. (2002). The global biogeography of roots. Ecological Monographs, 72(3), pp.311–328.
# Hay, R.K.M. (1995). Harvest index: a review of its use in plant breeding and crop production. Outlook on Agriculture, 24(1), pp.9–16.

import numpy as np
import pandas as pd

def root_litterinput(z, e_fold_depth):
    return 1 - np.exp(-z / e_fold_depth)

def fraction_litterinputs(upper, lower, e_fold_depth):
    """Compute the fraction of the root litter inputs processed between two depths."""
    upper_root_litter = root_litterinput(upper, e_fold_depth)
    lower_root_litter = root_litterinput(lower, e_fold_depth)
    return lower_root_litter - upper_root_litter

years = [2018, 2015, 2009]
for year in years:
    input_year_vals = []
    for idx, row in df.iterrows():
        # get NPP and LC
        npp_minus_0 = row[f'MODIS_NPP_{year}gps_{year-0}']
        npp_minus_1 = row[f'MODIS_NPP_{year}gps_{year-1}']
        npp_minus_2 = row[f'MODIS_NPP_{year}gps_{year-2}']
        npp_minus_3 = row[f'MODIS_NPP_{year}gps_{year-3}']
        npp_minus_4 = row[f'MODIS_NPP_{year}gps_{year-4}']
        npp = np.mean([npp_minus_0, npp_minus_1, npp_minus_2, npp_minus_3, npp_minus_4])
        lc = row[f'lc1_{year}']
        if pd.isna(lc) or pd.isna(npp):
            input_year_vals.append(np.nan)
            continue
        letter = lc[0]
        if letter in ['A', 'H']: # in case of artificial and bare land use grass land values (artificial probably still underestimates input as MODIS value lower than at site; bare land wouldn't work with 0 inputs)
            letter = 'E'
        # get root and shoot of NPP
        RS = RS_ef_HI[letter][0]
        root_npp = npp * RS / (1 + RS)
        shoot_npp = npp * 1 / (1 + RS)
        # get remains of shoot
        HI = RS_ef_HI[letter][2]
        remains = (1-HI) * shoot_npp
        # get roots 0-20
        efd = RS_ef_HI[letter][1]
        zero_20 = root_npp * fraction_litterinputs(0.0, 0.2, efd)
        # input
        input_val = zero_20 + remains
        input_year_vals.append(input_val)
    df[f'input_{year}'] = input_year_vals
        

/tmp/ipykernel_1589967/1485740418.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'input_{year}'] = input_year_vals
/tmp/ipykernel_1589967/1485740418.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'input_{year}'] = input_year_vals
/tmp/ipykernel_1589967/1485740418.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fra

Calculate 5 year means for MICi and MAOCi prediction

In [3]:
# calculate means of sampling year -5 for all 3 sampling years
mean_cols = {}
for year in [2009, 2015, 2018]:
    for var in ['Fluxcom_H', 'Fluxcom_LE', 'era5_land_t2m', 'era5_land_tp', 'era5_land_hpet', 'MODIS_NPP']:
        # get values of years -5
        var_minus_0 = df[f'{var}_{year}gps_{year-0}']
        var_minus_1 = df[f'{var}_{year}gps_{year-1}']
        var_minus_2 = df[f'{var}_{year}gps_{year-2}']
        var_minus_3 = df[f'{var}_{year}gps_{year-3}']
        var_minus_4 = df[f'{var}_{year}gps_{year-4}']
        # get mean
        mean_value = np.mean([var_minus_0, var_minus_1, var_minus_2, var_minus_3, var_minus_4], axis=0)
        # insert
        mean_cols[f'{var}_{year}-5_mean'] = mean_value
        print('added:', f'{var}_{year}-5_mean')
df = pd.concat([df, pd.DataFrame(mean_cols, index=df.index)], axis=1)

added: Fluxcom_H_2009-5_mean
added: Fluxcom_LE_2009-5_mean
added: era5_land_t2m_2009-5_mean
added: era5_land_tp_2009-5_mean
added: era5_land_hpet_2009-5_mean
added: MODIS_NPP_2009-5_mean
added: Fluxcom_H_2015-5_mean
added: Fluxcom_LE_2015-5_mean
added: era5_land_t2m_2015-5_mean
added: era5_land_tp_2015-5_mean
added: era5_land_hpet_2015-5_mean
added: MODIS_NPP_2015-5_mean
added: Fluxcom_H_2018-5_mean
added: Fluxcom_LE_2018-5_mean
added: era5_land_t2m_2018-5_mean
added: era5_land_tp_2018-5_mean
added: era5_land_hpet_2018-5_mean
added: MODIS_NPP_2018-5_mean


Climate changes for change prediction

In [4]:
from sklearn.linear_model import LinearRegression
for var in ['doy', 'Fluxcom_H', 'Fluxcom_LE', 'era5_land_t2m', 'era5_land_tp', 'era5_land_hpet', 'MODIS_NPP']:
    if var == 'doy':
        mask = (~df[f"{var}_2009"].isna()) & (~df[f"{var}_2015"].isna()) & (~df[f"{var}_2018"].isna())
    else:
        mask = (~df[f"{var}_2009-5_mean"].isna()) & (~df[f"{var}_2015-5_mean"].isna()) & (~df[f"{var}_2018-5_mean"].isna())
    years = np.array([2009, 2015, 2018]).reshape(-1, 1)
    lr_slopes = np.full(len(df), np.nan)
    lr_intercepts = np.full(len(df), np.nan)
    for idx in df[mask].index:
        if var == 'doy':
            y = df.loc[idx, [f"{var}_{y}" for y in years.flatten()]].values # get SOC values of the 3 years
        else:
            y = df.loc[idx, [f"{var}_{y}-5_mean" for y in years.flatten()]].values # get SOC values of the 3 years
        model = LinearRegression().fit(years, y) # Lienar regression on years (x) and SOC values (y)
        lr_slopes[idx], _ = model.coef_[0], model.intercept_

    df[f"{var}_linreg_slope"] = lr_slopes

    if var == 'doy':
        cols = [f"{var}_{year}" for year in [2009, 2015, 2018] if f"{var}_{year}" in df.columns]
    else:
        cols = [f"{var}_{year}-5_mean" for year in [2009, 2015, 2018] if f"{var}_{year}-5_mean" in df.columns]
    avg = np.full(len(df), np.nan)
    if len(cols) == 3:
        avg[mask] = df.loc[mask, cols].astype(float).mean(axis=1)
        df[f"{var}_avg_09_15_18"] = avg

/tmp/ipykernel_1589967/1054217767.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{var}_linreg_slope"] = lr_slopes
/tmp/ipykernel_1589967/1054217767.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{var}_avg_09_15_18"] = avg
/tmp/ipykernel_1589967/1054217767.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, us

In [5]:
df.to_pickle("4_with_nc.pkl")